# 🔬 Audit & Reconciliation: `gold.clinisys_embrioes_outcomes` vs `gold.planilha_embryoscope_combined`

This notebook provides a comprehensive audit and metric validation comparing the newly generated table **`gold.clinisys_embrioes_outcomes`** against the legacy table **`gold.planilha_embryoscope_combined`**.

### Key Audit Objectives:
1. **Volume & Cohort Scope**: Verify total embryos (~321,347) and breakdown by transfer status (`is_transferred`).
2. **Elimination of False Positives**: Validate that non-transferred/discarded/fertilization-failure oocytes no longer receive false pregnancy outcomes (fixing the legacy flaw where 21,661 non-transferred eggs falsely inherited FET outcomes).
3. **Transferred Embryo Outcome Enrichment**: Quantify the coverage of pregnancy results, clinical pregnancy, live births, and delivery dates on transferred embryos.
4. **Planilha vs REDLARA Quality Cross-Check**: Audit the two-way concordance between lab spreadsheets and the REDLARA registry on transferred embryos.
5. **Temporal & Regional Breakdown**: Track outcome completeness across years (2021–2026) and clinic units.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Resilient Database Path Resolution (supports running from root or notebooks directory)
possible_paths = [
    '../../database/huntington_data_lake.duckdb',
    '../database/huntington_data_lake.duckdb',
    'database/huntington_data_lake.duckdb',
    os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__ if '__file__' in locals() else os.getcwd()))), 'database', 'huntington_data_lake.duckdb')
]
DUCKDB_PATH = next((p for p in possible_paths if os.path.exists(p)), 'database/huntington_data_lake.duckdb')
con = duckdb.connect(DUCKDB_PATH, read_only=True)
print(f'DuckDB database connected at: {DUCKDB_PATH} (read-only)')

## 1. Volume & High-Level Comparison

In [ ]:
# Overall volumes
df_vol_new = con.execute("""
    SELECT 
        count(*) as total_embryos,
        sum(is_transferred) as transferred_embryos,
        count(*) - sum(is_transferred) as non_transferred_embryos,
        count(outcome_final_result) as has_final_result,
        count(outcome_final_gravidez_clinica) as has_final_grav_clin,
        count(outcome_final_no_nascidos) as has_final_newborns,
        count(outcome_final_data_parto) as has_final_delivery_date
    FROM gold.clinisys_embrioes_outcomes
""").fetchdf()

df_vol_old = con.execute("""
    SELECT 
        count(*) as total_embryos,
        count(fet_resultado) as has_fet_resultado,
        count(fet_gravidez_clinica) as has_fet_grav_clin,
        count(merged_numero_de_nascidos) as has_merged_newborns
    FROM gold.planilha_embryoscope_combined
""").fetchdf()

print('--- NEW Table (gold.clinisys_embrioes_outcomes) ---')
display(df_vol_new)

print('\n--- LEGACY Table (gold.planilha_embryoscope_combined) ---')
display(df_vol_old)

## 2. False Positive Audit: Discarded/Non-Transferred Oocytes

In the legacy pipeline, puncture-level joins caused entire puncture cohorts (including non-fertilized or discarded oocytes) to inherit outcomes from subsequent FET procedures. Below we audit outcomes by transfer status.

In [ ]:
# Audit outcomes on non-transferred embryos in NEW vs OLD table
df_fp_audit = con.execute("""
    WITH new_stats AS (
        SELECT 
            CASE WHEN is_transferred = 1 THEN 'Transferred Embryos' ELSE 'Non-Transferred / Discarded' END as embryo_cohort,
            count(*) as total_rows,
            count(outcome_final_result) as with_outcome_result,
            count(outcome_final_gravidez_clinica) as with_clinical_pregnancy,
            count(outcome_final_no_nascidos) as with_live_births
        FROM gold.clinisys_embrioes_outcomes
        GROUP BY 1
    ),
    old_stats AS (
        SELECT 
            CASE 
                WHEN (emb_cong_transferidos = 'Transferido' OR descong_em_DataTransferencia IS NOT NULL OR trat2_data_transferencia IS NOT NULL) 
                     OR (trat1_data_transferencia IS NOT NULL AND (trat1_motivo_nao_transferir IS NULL OR trat1_motivo_nao_transferir = '') AND (trat1_resultado_tratamento IS NULL OR trat1_resultado_tratamento NOT IN ('No transfer', 'Congelamento de óvulos', 'Congelamento de vulos'))) THEN 'Transferred Embryos'
                ELSE 'Non-Transferred / Discarded'
            END as embryo_cohort,
            count(*) as old_total_rows,
            count(fet_resultado) as old_with_fet_resultado,
            count(fet_gravidez_clinica) as old_with_clinical_pregnancy,
            count(merged_numero_de_nascidos) as old_with_live_births
        FROM gold.planilha_embryoscope_combined
        GROUP BY 1
    )
    SELECT 
        n.embryo_cohort,
        n.total_rows as new_total_rows,
        n.with_outcome_result as new_with_outcome,
        n.with_clinical_pregnancy as new_with_clin_preg,
        n.with_live_births as new_with_live_births,
        o.old_total_rows,
        o.old_with_fet_resultado as legacy_with_outcome,
        o.old_with_clinical_pregnancy as legacy_with_clin_preg,
        o.old_with_live_births as legacy_with_live_births
    FROM new_stats n
    JOIN old_stats o ON n.embryo_cohort = o.embryo_cohort;
""").fetchdf()

print('=== COMPARATIVE FALSE POSITIVE & COHORT AUDIT ===')
display(df_fp_audit)

## 3. Transferred Embryos Outcome Enrichment Metrics

Focusing strictly on **transferred embryos (24,664 embryos)**, we evaluate the coverage of outcomes across the independent sources.

In [ ]:
df_transf_metrics = con.execute("""
    SELECT 
        count(*) as total_transferred_embryos,
        sum(planilha_matched) as matched_in_planilha,
        sum(redlara_matched) as matched_in_redlara,
        sum(CASE WHEN planilha_matched = 1 OR redlara_matched = 1 THEN 1 ELSE 0 END) as matched_in_at_least_one,
        sum(CASE WHEN planilha_matched = 1 AND redlara_matched = 1 THEN 1 ELSE 0 END) as matched_in_both,
        count(outcome_final_result) as with_pregnancy_result,
        count(outcome_final_gravidez_clinica) as with_clinical_pregnancy,
        count(outcome_final_no_nascidos) as with_newborn_count,
        count(outcome_final_data_parto) as with_delivery_date
    FROM gold.clinisys_embrioes_outcomes
    WHERE is_transferred = 1;
""").fetchdf()

print('=== TRANSFERRED EMBRYOS ENRICHMENT METRICS ===')
display(df_transf_metrics.T.rename(columns={0: 'Count / Metric'}))


## 4. Dual-Source Quality Cross-Check & Concordance Matrix

For embryos matched in both Planilha lab spreadsheets and REDLARA registry, we examine the concordance on clinical pregnancy.

In [ ]:
df_crosscheck = con.execute("""
    SELECT 
        crosscheck_status,
        count(*) as transferred_embryos_count,
        round(count(*) * 100.0 / (SELECT count(*) FROM gold.clinisys_embrioes_outcomes WHERE is_transferred = 1), 2) as pct_of_transferred
    FROM gold.clinisys_embrioes_outcomes
    WHERE is_transferred = 1
    GROUP BY 1
    ORDER BY 2 DESC;
""").fetchdf()

print('=== QUALITY CROSS-CHECK CONCORDANCE STATUS ===')
display(df_crosscheck)

# Agreement Rate among dual-recorded cases
df_dual_agreement = con.execute("""
    SELECT 
        planilha_gravidez_clinica as planilha_preg,
        redlara_gravidez_clinica as redlara_preg,
        count(*) as count
    FROM gold.clinisys_embrioes_outcomes
    WHERE is_transferred = 1 
      AND planilha_gravidez_clinica IS NOT NULL 
      AND redlara_gravidez_clinica IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1, 2;
""").fetchdf()

print('\n=== DUAL CLINICAL PREGNANCY CONCORDANCE MATRIX ===')
display(df_dual_agreement)

## 5. Temporal Breakdown (2021 - 2026)

In [ ]:
df_annual = con.execute("""
    SELECT 
        proc_year as year,
        count(*) as total_transferred,
        sum(planilha_matched) as planilha_matched,
        sum(redlara_matched) as redlara_matched,
        sum(CASE WHEN planilha_matched = 1 OR redlara_matched = 1 THEN 1 ELSE 0 END) as any_source_matched,
        round(sum(CASE WHEN planilha_matched = 1 OR redlara_matched = 1 THEN 1 ELSE 0 END) * 100.0 / count(*), 1) as match_rate_pct,
        count(outcome_final_result) as with_outcome,
        count(outcome_final_gravidez_clinica) as with_clin_preg
    FROM gold.clinisys_embrioes_outcomes
    WHERE is_transferred = 1 AND proc_year BETWEEN 2021 AND 2026
    GROUP BY 1
    ORDER BY 1;
""").fetchdf()

print('=== ANNUAL MATCHING & OUTCOME PERFORMANCE ===')
display(df_annual)

## 6. Closing Connection

In [ ]:
con.close()
print('DuckDB connection closed successfully.')